# Train and validate NicheTrans with schema-compliant H5MU files

This notebook consumes two H5MU subsets that were prepared in advance: one for training and one for periodic validation/model selection. It can run a baseline model, a static gene-prior model, or both on a shared prior-covered gene panel for a fair comparison. Edit the paths and settings in the configuration cell before running.

In [1]:
from copy import copy
from pathlib import Path
import os
import warnings

import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from joblib import Parallel, delayed
from torch.optim import lr_scheduler

from args.args_h5mu import generate_args
from datasets.h5mu_dataset import H5MuDataManager, validate_h5mu
from model.nicheTrans import NicheTrans
from prior_AddOn.gene_embedding_loader import load_static_gene_prior
from prior_AddOn.gene_prior_filter import filter_dataset_by_gene_prior
from utils.utils import set_seed
from utils.utils_h5mu_dataloader import h5mu_dataloader
from utils.utils_training_h5mu import (
    build_criterion,
    evaluate,
    fit,
    infer_task_type,
    resolve_device,
)

## Configuration

Replace both placeholder paths. Set `EXPERIMENT_MODE` to `baseline`, `prior`, or `compare`. In `compare` mode, both models use the same prior-covered source-gene panel so the measured difference isolates the prior module rather than a change in input dimensionality. With `PARALLEL_COMPARE=True`, two independent worker processes are launched; `(0, 0)` shares one GPU and `(0, 1)` uses two GPUs. Set it to `False` for sequential execution. `task=auto` maps target `value_type=binary` to binary classification and all other value types to regression.

In [2]:
TRAIN_H5MU = Path(r"/home1/shezixi/data/isCDC/traing_set.h5mu")
TEST_H5MU = Path(r"/home1/shezixi/data/isCDC/testing_set.h5mu")

# Direct notebook switches for baseline/prior experiments.
EXPERIMENT_MODE = "baseline"  # baseline | prior | compare
PRIOR_MODEL = "scgpt"  # scgpt | geneformer
PRIOR_ROOT = Path("prior_AddOn/gene_embeddings")
PRIOR_ALLOW_NETWORK = False
PRIOR_NORMALIZE_EMBEDDING = True
PARALLEL_COMPARE = True
PARALLEL_GPU_IDS = (0, 0)  # same GPU; use (0, 1) for two GPUs

EXPERIMENT_MODE = str(EXPERIMENT_MODE).strip().lower()
PRIOR_MODEL = str(PRIOR_MODEL).strip().lower()
if EXPERIMENT_MODE not in {"baseline", "prior", "compare"}:
    raise ValueError("EXPERIMENT_MODE must be 'baseline', 'prior', or 'compare'.")
if PRIOR_MODEL not in {"scgpt", "geneformer"}:
    raise ValueError("PRIOR_MODEL must be 'scgpt' or 'geneformer'.")
if len(PARALLEL_GPU_IDS) != 2:
    raise ValueError("PARALLEL_GPU_IDS must contain exactly two GPU indices.")
if any(int(gpu_id) < 0 for gpu_id in PARALLEL_GPU_IDS):
    raise ValueError("PARALLEL_GPU_IDS must contain non-negative GPU indices.")

args = generate_args([
    "--train-path", str(TRAIN_H5MU),
    "--test-path", str(TEST_H5MU),
    "--source-modality", "rna",
    "--target-modality", "protein",
    "--n-neighbors", "12",
    "--preprocess", "auto",
    "--task", "auto",
    "--max-epoch", "40",
    "--eval-step", "1",
    "--train-batch", "32",
    "--test-batch", "32",
    "--workers", "4",
    "--device", "auto",
    "--output-dir", "outputs/h5mu",
    "--run-name", "nichetrans_h5mu",
])
display({
    **vars(args),
    "experiment_mode": EXPERIMENT_MODE,
    "prior_model": PRIOR_MODEL,
    "prior_root": str(PRIOR_ROOT),
    "prior_allow_network": PRIOR_ALLOW_NETWORK,
    "prior_normalize_embedding": PRIOR_NORMALIZE_EMBEDDING,
    "parallel_compare": PARALLEL_COMPARE,
    "parallel_gpu_ids": PARALLEL_GPU_IDS,
})

{'train_path': '/home1/shezixi/data/isCDC/traing_set.h5mu',
 'test_path': '/home1/shezixi/data/isCDC/testing_set.h5mu',
 'source_modality': 'rna',
 'target_modality': 'protein',
 'n_neighbors': 12,
 'preprocess': 'auto',
 'rna_target_sum': 1000.0,
 'task': 'auto',
 'workers': 4,
 'noise_rate': 0.2,
 'dropout_rate': 0.2,
 'neighbor_mask_probability': 0.3,
 'neighbor_keep_probability': 0.5,
 'max_epoch': 40,
 'eval_step': 1,
 'train_batch': 32,
 'test_batch': 32,
 'optimizer': 'adam',
 'lr': 0.0003,
 'weight_decay': 0.0005,
 'stepsize': 20,
 'gamma': 0.1,
 'seed': 1,
 'device': 'auto',
 'output_dir': 'outputs/h5mu',
 'run_name': 'nichetrans_h5mu',
 'experiment_mode': 'baseline',
 'prior_model': 'scgpt',
 'prior_root': 'prior_AddOn/gene_embeddings',
 'prior_allow_network': False,
 'prior_normalize_embedding': True,
 'parallel_compare': True,
 'parallel_gpu_ids': (0, 0)}

## Validate, load, and prepare the source panel

Feature names and order must be identical between the two files. Spatial neighbors are built independently inside each file and `sample_id`. For prior-enabled runs, the organism is read from H5MU metadata and the source panel is filtered before any DataLoader is created.

In [3]:
for path, label in [(Path(args.train_path), "training"), (Path(args.test_path), "testing")]:
    if not path.is_file():
        raise FileNotFoundError(f"Set the {label} H5MU path in the configuration cell: {path}")

train_metadata = validate_h5mu(args.train_path)
test_metadata = validate_h5mu(args.test_path)
dataset = H5MuDataManager(
    train_path=args.train_path,
    test_path=args.test_path,
    source_modality=args.source_modality,
    target_modality=args.target_modality,
    n_neighbors=args.n_neighbors,
    preprocess=args.preprocess,
    rna_target_sum=args.rna_target_sum,
)

train_value_type = dataset.train_assays[args.target_modality]["value_type"]
test_value_type = dataset.test_assays[args.target_modality]["value_type"]
task = infer_task_type(train_value_type, test_value_type, requested=args.task)
uses_prior = EXPERIMENT_MODE in {"prior", "compare"}
prior_species = None
priors = None
prior_filter_info = None
original_source_length = dataset.source_length

def canonical_prior_species(value):
    key = str(value).strip().lower().replace("-", "_").replace(" ", "_")
    if key in {"human", "homo_sapiens", "hsapiens", "hs"}:
        return "human"
    if key in {"mouse", "mus_musculus", "mmusculus", "mm", "murine"}:
        return "mouse"
    raise ValueError(
        "Static gene priors currently support human or mouse H5MU datasets; "
        f"got organism={value!r}."
    )

if uses_prior:
    if str(args.source_modality).lower() != "rna":
        raise ValueError(
            "Static gene priors require source_modality='rna'. "
            f"Current source modality: {args.source_modality!r}."
        )
    if not PRIOR_ROOT.is_dir():
        raise FileNotFoundError(f"Static gene-prior root was not found: {PRIOR_ROOT.resolve()}")

    train_species = canonical_prior_species(dataset.train_database["organism"])
    test_species = canonical_prior_species(dataset.test_database["organism"])
    if train_species != test_species:
        raise ValueError(
            "Training and testing H5MU organisms must match for prior alignment; "
            f"got {dataset.train_database['organism']!r} and "
            f"{dataset.test_database['organism']!r}."
        )
    prior_species = train_species
    dataset_key = "__".join([
        str(dataset.train_database["dataset_id"]),
        str(dataset.test_database["dataset_id"]),
    ])
    priors = load_static_gene_prior(
        source_panel=dataset.source_panel,
        species=prior_species,
        models=(PRIOR_MODEL,),
        root=PRIOR_ROOT,
        dataset_key=dataset_key,
        allow_network=PRIOR_ALLOW_NETWORK,
    )
    dataset, priors, prior_filter_info = filter_dataset_by_gene_prior(
        dataset=dataset,
        priors=priors,
        prior_model=PRIOR_MODEL,
    )
    coverage_before = prior_filter_info.get("coverage_before") or {}
    coverage_after = prior_filter_info.get("coverage_after") or {}
    print(
        f"Gene prior preparation ({PRIOR_MODEL}, {prior_species}): kept "
        f"{dataset.source_length}/{original_source_length} genes; "
        f"removed {len(prior_filter_info['removed_genes'])}; "
        f"coverage {coverage_before.get('coverage', float('nan')):.2%} -> "
        f"{coverage_after.get('coverage', float('nan')):.2%}."
    )

source_panel_policy = (
    "prior_covered_shared" if EXPERIMENT_MODE == "compare"
    else "prior_covered" if EXPERIMENT_MODE == "prior"
    else "full"
)
print(f"Task: {task}; source: {dataset.source_length}; target: {dataset.target_length}")
print(f"Training observations: {len(dataset.training):,}")
print(f"Validation observations: {len(dataset.testing):,}")
print(f"Source-panel policy: {source_panel_policy}")

Task: regression; source: 405; target: 27
Training observations: 221,139
Validation observations: 244,395
Source-panel policy: full


## Experiment variants and model builders

The baseline disables prior pooling. The prior variant uses QKV pooling with the aligned static embeddings. Both variants retain the same standard NicheTrans backbone and target heads.

In [4]:
default_device = resolve_device(args.device)
criterion = build_criterion(task)
if dataset.target_length > 512:
    warnings.warn(
        f"The standard NicheTrans model has {dataset.target_length} target heads; "
        "training may be slow or memory intensive."
    )

if EXPERIMENT_MODE == "baseline":
    experiment_variants = ["baseline"]
elif EXPERIMENT_MODE == "prior":
    experiment_variants = ["prior"]
else:
    experiment_variants = ["baseline", "prior"]

def build_model(variant, run_dataset, run_device, allow_data_parallel=True):
    use_prior = variant == "prior"
    model = NicheTrans(
        source_length=run_dataset.source_length,
        target_length=run_dataset.target_length,
        noise_rate=args.noise_rate,
        dropout_rate=args.dropout_rate,
        priors=priors if use_prior else None,
        prior_model=PRIOR_MODEL if use_prior else None,
        prior_pooling_mode="qkv" if use_prior else "none",
        normalize_prior_embedding=PRIOR_NORMALIZE_EMBEDDING if use_prior else False,
    ).to(run_device)
    if allow_data_parallel and run_device.type == "cuda" and torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    return model

def build_optimizer(model):
    if args.optimizer == "adam":
        return torch.optim.Adam(
            model.parameters(), lr=args.lr, weight_decay=args.weight_decay
        )
    return torch.optim.SGD(
        model.parameters(), lr=args.lr, weight_decay=args.weight_decay
    )

def build_scheduler(optimizer):
    if args.stepsize <= 0:
        return None
    return lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

print(
    f"Device: {default_device}; loss: {type(criterion).__name__}; "
    f"variants: {', '.join(experiment_variants)}"
)

Device: cuda; loss: MSELoss; variants: baseline


## Train each variant with periodic validation

Each run independently opens the H5MU files and rebuilds its DataLoaders, model, optimizer, and scheduler. Parallel comparison uses two separate GPU processes, which may share one GPU when `PARALLEL_GPU_IDS=(0, 0)`; ensure that GPU memory can hold both runs. The common seed is set before model construction and again immediately before training so model initialization, shuffled sample order, and random neighbor masking remain comparable. Regression runs maximize mean Pearson correlation; binary runs maximize mean AUROC.

In [5]:
output_dir = Path(args.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

prior_filter_metadata = None
if prior_filter_info is not None:
    prior_filter_metadata = {
        "original_source_panel": [str(value) for value in prior_filter_info["original_source_panel"]],
        "filtered_source_panel": [str(value) for value in prior_filter_info["filtered_source_panel"]],
        "removed_genes": [str(value) for value in prior_filter_info["removed_genes"]],
        "coverage_before": prior_filter_info.get("coverage_before"),
        "coverage_after": prior_filter_info.get("coverage_after"),
    }

def build_worker_dataset():
    run_dataset = H5MuDataManager(
        train_path=args.train_path,
        test_path=args.test_path,
        source_modality=args.source_modality,
        target_modality=args.target_modality,
        n_neighbors=args.n_neighbors,
        preprocess=args.preprocess,
        rna_target_sum=args.rna_target_sum,
    )
    if prior_filter_info is not None:
        run_dataset.select_source_features(prior_filter_info["keep_mask"])
    return run_dataset

def resolve_experiment_device(assigned_gpu_id):
    if assigned_gpu_id is None:
        return resolve_device(args.device)
    assigned_gpu_id = int(assigned_gpu_id)
    if not torch.cuda.is_available():
        raise RuntimeError("Parallel comparison requested but CUDA is unavailable.")
    if assigned_gpu_id >= torch.cuda.device_count():
        raise ValueError(
            f"GPU {assigned_gpu_id} is not visible; found {torch.cuda.device_count()} CUDA devices."
        )
    torch.cuda.set_device(assigned_gpu_id)
    return torch.device(f"cuda:{assigned_gpu_id}")

def run_experiment(variant, assigned_gpu_id=None):
    use_prior = variant == "prior"
    run_label = f"prior_{PRIOR_MODEL}" if use_prior else "baseline"
    run_name = f"{args.run_name}_{run_label}"
    checkpoint_path = output_dir / f"{run_name}_best.pth"
    history_path = output_dir / f"{run_name}_history.csv"
    metrics_path = output_dir / f"{run_name}_validation_metrics.csv"
    run_device = resolve_experiment_device(assigned_gpu_id)
    process_id = os.getpid()
    print(
        f"\n==> Starting {run_label} with seed {args.seed}; "
        f"PID={process_id}; device={run_device}"
    )

    set_seed(args.seed)
    run_dataset = build_worker_dataset()
    loader_args = copy(args)
    if assigned_gpu_id is not None:
        # Avoid unsupported nested multiprocessing inside a loky worker.
        loader_args.workers = 0
    loader_workers = int(loader_args.workers)
    trainloader, validationloader = h5mu_dataloader(loader_args, run_dataset)
    print(f"DataLoader workers for {run_label}: {loader_workers}")
    model = build_model(
        variant,
        run_dataset=run_dataset,
        run_device=run_device,
        allow_data_parallel=assigned_gpu_id is None,
    )
    run_criterion = build_criterion(task)
    optimizer = build_optimizer(model)
    scheduler = build_scheduler(optimizer)

    checkpoint_metadata = {
        "args": vars(args),
        "experiment": {
            "mode": EXPERIMENT_MODE,
            "variant": run_label,
            "seed": args.seed,
            "process_id": process_id,
            "device": str(run_device),
            "assigned_gpu_id": assigned_gpu_id,
            "dataloader_workers": loader_workers,
            "source_panel_policy": source_panel_policy,
        },
        "prior": {
            "enabled": use_prior,
            "selected_model": PRIOR_MODEL if uses_prior else None,
            "pooling_mode": "qkv" if use_prior else "none",
            "species": prior_species,
            "normalize_embedding": PRIOR_NORMALIZE_EMBEDDING if use_prior else False,
            "filter": prior_filter_metadata,
        },
        "source_panel": [str(value) for value in run_dataset.source_panel],
        "target_panel": [str(value) for value in run_dataset.target_panel],
        "train_database": run_dataset.train_database,
        "test_database": run_dataset.test_database,
        "train_assays": run_dataset.train_assays,
        "test_assays": run_dataset.test_assays,
    }

    # Reset after variant-specific layer initialization so data order and augmentation match.
    set_seed(args.seed)
    result = fit(
        model=model,
        trainloader=trainloader,
        validationloader=validationloader,
        optimizer=optimizer,
        criterion=run_criterion,
        device=run_device,
        task=task,
        max_epochs=args.max_epoch,
        eval_step=args.eval_step,
        checkpoint_path=checkpoint_path,
        scheduler=scheduler,
        target_panel=run_dataset.target_panel,
        checkpoint_metadata=checkpoint_metadata,
        neighbor_mask_probability=args.neighbor_mask_probability,
        neighbor_keep_probability=args.neighbor_keep_probability,
    )
    final_evaluation = evaluate(
        model=model,
        dataloader=validationloader,
        criterion=run_criterion,
        device=run_device,
        task=task,
        target_panel=run_dataset.target_panel,
    )

    pd.DataFrame(result["history"]).to_csv(history_path, index=False)
    pd.DataFrame(final_evaluation["per_feature"]).to_csv(metrics_path, index=False)
    record = {
        "variant": run_label,
        "use_prior": use_prior,
        "prior_model": PRIOR_MODEL if use_prior else None,
        "seed": args.seed,
        "source_features": run_dataset.source_length,
        "removed_source_features": original_source_length - run_dataset.source_length,
        "process_id": process_id,
        "device": str(run_device),
        "assigned_gpu_id": assigned_gpu_id,
        "dataloader_workers": loader_workers,
        "best_epoch": result["best_epoch"],
        "checkpoint": result["checkpoint_path"],
        "history_csv": str(history_path.resolve()),
        "per_feature_csv": str(metrics_path.resolve()),
        **final_evaluation["summary"],
    }
    artifacts = {
        "training": result,
        "evaluation": final_evaluation,
    }
    print(f"Finished {run_label}; best epoch: {result['best_epoch']}")
    print(f"Checkpoint: {result['checkpoint_path']}")
    print(f"History: {history_path.resolve()}")
    print(f"Per-feature metrics: {metrics_path.resolve()}")

    run_dataset.close()
    del model, optimizer, scheduler, run_criterion, trainloader, validationloader
    if run_device.type == "cuda":
        torch.cuda.empty_cache()
    return record, artifacts

requested_parallel_compare = EXPERIMENT_MODE == "compare" and PARALLEL_COMPARE
parallel_gpu_ids = tuple(int(gpu_id) for gpu_id in PARALLEL_GPU_IDS)
parallel_gpu_ready = (
    requested_parallel_compare
    and str(args.device).lower() != "cpu"
    and torch.cuda.is_available()
    and all(gpu_id < torch.cuda.device_count() for gpu_id in parallel_gpu_ids)
)
if requested_parallel_compare and not parallel_gpu_ready:
    warnings.warn(
        "Parallel comparison requested, but the configured CUDA devices are unavailable; "
        "falling back to sequential execution."
    )

if parallel_gpu_ready:
    print(
        "Launching independent GPU processes: "
        + ", ".join(
            f"{variant}->cuda:{gpu_id}"
            for variant, gpu_id in zip(experiment_variants, parallel_gpu_ids)
        )
    )
    experiment_outputs = Parallel(n_jobs=2, backend="loky")(
        delayed(run_experiment)(variant, assigned_gpu_id=gpu_id)
        for variant, gpu_id in zip(experiment_variants, parallel_gpu_ids)
    )
else:
    experiment_outputs = [
        run_experiment(variant) for variant in experiment_variants
    ]

experiment_records = []
experiment_artifacts = {}
for record, artifacts in experiment_outputs:
    experiment_records.append(record)
    experiment_artifacts[record["variant"]] = artifacts


==> Starting baseline with seed 1; PID=3260508; device=cuda
DataLoader workers for baseline: 4
Epoch 1/40 - train_loss=1.290759 - validation_loss=1.256899 - pearson_mean=0.622682 - best
Epoch 2/40 - train_loss=1.169187 - validation_loss=1.241379 - pearson_mean=0.638513 - best
Epoch 3/40 - train_loss=1.148988 - validation_loss=1.295422 - pearson_mean=0.637955
Epoch 4/40 - train_loss=1.139339 - validation_loss=1.242871 - pearson_mean=0.646473 - best
Epoch 5/40 - train_loss=1.132748 - validation_loss=1.244620 - pearson_mean=0.643398
Epoch 6/40 - train_loss=1.126675 - validation_loss=1.300508 - pearson_mean=0.644601
Epoch 7/40 - train_loss=1.124241 - validation_loss=1.267264 - pearson_mean=0.638973
Epoch 8/40 - train_loss=1.120850 - validation_loss=1.255684 - pearson_mean=0.644085
Epoch 9/40 - train_loss=1.120164 - validation_loss=1.237803 - pearson_mean=0.650628 - best
Epoch 10/40 - train_loss=1.115746 - validation_loss=1.245294 - pearson_mean=0.643684
Epoch 11/40 - train_loss=1.113966 -

## Compare restored-best validation results

The comparison table contains one row per variant. When both variants are present, each `*_delta_vs_baseline` column is computed as `variant - baseline`; higher is better for correlation/AUROC, while lower is better for loss/RMSE.

In [6]:
comparison_df = pd.DataFrame(experiment_records)
metric_columns = (
    ["loss", "auroc_mean"]
    if task == "binary"
    else ["loss", "pearson_mean", "spearman_mean", "rmse_mean"]
)
baseline_rows = comparison_df.loc[comparison_df["variant"] == "baseline"]
if len(baseline_rows) == 1 and bool(comparison_df["use_prior"].any()):
    baseline_row = baseline_rows.iloc[0]
    for metric in metric_columns:
        if metric in comparison_df.columns:
            comparison_df[f"{metric}_delta_vs_baseline"] = (
                comparison_df[metric] - baseline_row[metric]
            )

comparison_path = output_dir / f"{args.run_name}_{EXPERIMENT_MODE}_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
display_columns = [
    "variant",
    "use_prior",
    "prior_model",
    "seed",
    "source_features",
    "removed_source_features",
    "process_id",
    "device",
    "assigned_gpu_id",
    "dataloader_workers",
    "best_epoch",
]
display_columns += [column for column in metric_columns if column in comparison_df.columns]
display_columns += [
    column for column in comparison_df.columns if column.endswith("_delta_vs_baseline")
]
print(f"Comparison: {comparison_path.resolve()}")
display(comparison_df[display_columns])

for run_label, artifacts in experiment_artifacts.items():
    print(f"{run_label} per-feature metrics (first 20):")
    display(pd.DataFrame(artifacts["evaluation"]["per_feature"]).head(20)) 

Comparison: /yzyStorage/home/shezixi/code/NicheEmbedding/BINN/outputs/h5mu/nichetrans_h5mu_baseline_comparison.csv


,variant,use_prior,prior_model,seed,source_features,removed_source_features,process_id,device,assigned_gpu_id,dataloader_workers,best_epoch,loss,pearson_mean,spearman_mean,rmse_mean
0,baseline,False,None,1,405,0,3260508,cuda,None,4,35,1.232449,0.654545,0.564383,1.061084


baseline per-feature metrics (first 20):


,feature,pearson,spearman,rmse,n_valid
0,PD-1,0.723736,0.526641,0.857970,244395
1,VISTA,0.532297,0.521190,1.528188,244395
2,PD-L1,0.295543,0.260320,0.578184,244395
3,LAG-3,0.766220,0.549088,1.071322,244395
4,CD16,0.757092,0.729828,1.416124,244395
5,GranzymeB,0.467267,0.233533,0.437678,244395
6,CD163,0.766119,0.701342,1.234093,244395
7,CD4,0.711196,0.590996,1.043165,244395
8,CD20,0.912959,0.598639,0.713067,244395
9,CD8A,0.783447,0.724463,1.426843,244395


## Close backed H5MU handles

Run this cell when training finishes or before changing input files.

In [7]:
dataset.close()
print("H5MU file handles closed.")

H5MU file handles closed.
